# 17. Monitorización, mantenimiento y reentrenamiento

**Fases del guía metodológica cubiertas: 23 (Monitorización, mantenimiento y reentrenamiento)**



## 23.1 Monitorización técnica

| Señal | Umbral/alerta |
|---|---|
| Disponibilidad de la API | < 99 % en ventana semanal |
| Latencia p95 | > 200 ms |
| Errores HTTP 4xx/5xx | > 1 % |
| Memoria/CPU | > 80 % sostenido |

## 23.2 Monitorización de datos (drift)

Con cada nueva cohorte se comparan las features entrantes frente a las de entrenamiento:
- **PSI (Population Stability Index)** por feature: > 0.2 -> alerta.
- Categorías nuevas no vistas -> alerta (la API las rechaza).
- Distribución de probabilidades predichas: cambio > 20 % relativo -> alerta.

### 23.2.1 PSI con la utilidad del proyecto

El cálculo de PSI está centralizado en `src/evaluation/monitoring.py` (función `psi`).
Aquí simulamos una cohorte nueva con drift en `absences` y `goout` y evaluamos el PSI de
tres features: valores < 0.1 indican estabilidad, 0.1-0.2 cambio moderado y > 0.2 alerta
severa. Esta es la misma utilidad que usaría un proceso de monitorización en producción.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from src.evaluation.monitoring import psi, check_drift, get_abstention_zone

from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
d = load_processed()
Xtr = add_domain_features(d["X_train"])

# Simulación: cohorte nueva con shift en absences y goout
rng = np.random.default_rng(7)
Xnew = Xtr.sample(n=200, random_state=7).copy()
Xnew["absences"] = Xnew["absences"] + rng.normal(0, 3, 200).clip(0, None)
Xnew["goout"] = np.clip(Xnew["goout"] + rng.choice([0, 1], 200, p=[0.7, 0.3]), 1, 5)

for c in ["goout", "absences", "age"]:
    r = check_drift(Xtr[c].values, Xnew[c].values, c)
    print(f"PSI {c:10s}: {r['psi']:.3f}  {'<-- ALERTA' if r['alerta'] else ''}")
print("\nZona de abstención configurada:", get_abstention_zone())


PSI goout     : 0.097  
PSI absences  : 0.105  
PSI age       : 0.008  

Zona de abstención configurada: [0.3, 0.6]



## 23.3 Monitorización de rendimiento

Cuando lleguen etiquetas reales (encuestas de la nueva cohorte):
- Recalcular ROC-AUC, PR-AUC, F1, Brier por trimestre.
- Métricas por subgrupo (sexo, escuela, edad).
- Tasa de abstención y comparación con el umbral de coste.

## 23.4 Reentrenamiento

**Criterios para reentrenar** (cualquiera dispara el proceso):
1. PSI > 0.2 en >= 2 features clave, o
2. ROC-AUC en datos nuevos < 0.70 durante 2 trimestres, o
3. Nuevo dataset oficial con más cohortes (>= 500 registros nuevos).

**Proceso** (documentado en `docs/model_card.md`):
```
Nuevos datos -> auditoría (igual que fase 3) -> split nuevo con test bloqueado NUEVO
-> entrenar candidatos -> comparar contra modelo actual en el MISMO test nuevo
-> aprobación -> versionado v2 -> rollback automático si degrada
```

- **Versionado**: `models/final_model_v{version}.joblib` + `models/registry.json`.
- **Rollback**: el artefacto anterior se conserva siempre.
- **Registro de cambios**: `docs/CHANGELOG.md`.

### 23.4.1 Registro de versiones sin duplicados

Usamos `register_model_version` (de `src/evaluation/monitoring.py`), que **deduplica**
por (versión, modelo): si ya existe una entrada, la actualiza en lugar de añadir otra.
Esto corrige el problema de registros duplicados detectado en la auditoría. Ejecutamos
el registro de la versión actual y mostramos el fichero resultante.


In [2]:

# Utilidad de registro de versiones de modelo (fase 23.4)
from src.evaluation.monitoring import register_model_version
registry = register_model_version(
    version="v1",
    model_name="RandomForest",
    roc_auc_test=0.766,
    notes="RandomForest tuneado balanced + calibración sigmoidal. Dataset 662 alumnos. Umbral de coste 0.34.",
)
print((ROOT / "models" / "registry.json").read_text(encoding="utf-8"))


[
  {
    "version": "v1",
    "model": "RandomForest",
    "date": "2026-08-14",
    "roc_auc_test": 0.766,
    "status": "en_produccion",
    "notes": "RandomForest tuneado balanced + calibraci\u00f3n sigmoidal. Dataset 662 alumnos. Umbral de coste 0.34."
  }
]
